# Program Pengujian: Mengenali 1 Gambar

Alur: *upload* 1 gambar, baca berkas bobot, hitung umpan maju saja, lalu tampilkan apakah gambar dikenali atau tidak.
Mengikuti *Application procedure* Fausett, *Fundamentals of Neural Networks* (1994), hlm. 299 sampai 300.

Jalankan sel dari atas ke bawah. Sel 4 akan meminta Anda memilih 1 gambar.

## Sel 1. Pengaturan Lingkungan dan Ambang Batas

* Menghubungkan Google Drive, memuat daftar 3 nama kelas, serta menentukan batas ambang keyakinan minimum (*threshold* = 0.5) agar citra dinyatakan dikenali.

In [ ]:
import io
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import drive, files

drive.mount('/content/drive')

BERKAS_BOBOT = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI/BOBOT/BOBOT.npz'
AMBANG_DIKENALI = 0.5    # nilai output terbesar minimal 0,5 agar dianggap dikenali
NAMA_KELAS = ['PESAWAT', 'MOBIL', 'KAPAL']

# Isi bila Anda tahu kelas gambar yang akan diunggah (0 = pesawat, 1 = mobil, 2 = kapal).
# Biarkan None bila tidak ingin dibandingkan.
KELAS_SEBENARNYA = None

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Berkas bobot    : {BERKAS_BOBOT}')
print(f'Ambang dikenali : {AMBANG_DIKENALI}')
print(f'Kelas           : {", ".join(NAMA_KELAS)}')

## Sel 2. Pemuatan Bobot Hasil Pelatihan

* Membaca berkas `BOBOT.npz` dari Drive (atau menyediakan dialog unggah berkas jika belum tersedia) dan memvalidasi kecocokan dimensi jaringan ($1024 \times 100 \times 3$).

In [ ]:
if os.path.exists(BERKAS_BOBOT):
    bobot = np.load(BERKAS_BOBOT)
    sumber = 'Google Drive'
else:
    print('Berkas bobot tidak ada di Drive. Silakan unggah BOBOT.npz.')
    unggahan = files.upload()
    bobot = np.load(io.BytesIO(unggahan[list(unggahan.keys())[0]]))
    sumber = 'unggahan'

v   = bobot['v']
v_0 = bobot['v_0']
w   = bobot['w']
w_0 = bobot['w_0']
LEBAR, TINGGI = [int(a) for a in bobot['ukuran']]

n = v.shape[0]   # node input
p = v.shape[1]   # node hidden
m = w.shape[1]   # node output

print('=' * 52)
print(f'BOBOT DIBACA (dari {sumber})')
print('=' * 52)
print(f'{"Variabel":<10}{"Ukuran":<14}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}')
print(f'\nNode input (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p): {p}')
print(f'Node output (m): {m}')
assert m == 3 and n == LEBAR * TINGGI, 'Ukuran bobot tidak sesuai'

## Sel 3. Fungsi Prapemrosesan dan Umpan Maju (Inferensi)

* Menyiapkan fungsi aktivasi sigmoid, fungsi `praproses` yang mereduksi citra menjadi vektor ternormalisasi skala 0 sampai 1, serta fungsi `umpan_maju` satu arah murni tanpa koreksi bobot.

In [ ]:
def f(x):
    return 1.0 / (1.0 + np.exp(-x))


def praproses(gambar):
    """Gambar PIL (RGB) -> vektor x berukuran n, nilai 0 sampai 1."""
    abu = gambar.convert('L')
    if abu.size != (LEBAR, TINGGI):
        abu = abu.resize((LEBAR, TINGGI))
    return np.asarray(abu, dtype=np.float64).flatten() / 255.0


def umpan_maju(x):
    z_in = v_0 + x @ v      # node hidden
    z    = f(z_in)
    y_in = w_0 + z @ w      # node output
    y    = f(y_in)
    return y

print('Fungsi siap: praproses(gambar) dan umpan_maju(x)')

## Sel 4. Pengunggahan Citra Uji

* Menampilkan tombol pemilih berkas interaktif untuk memasukkan satu gambar eksternal yang ingin diuji, lalu membaca dimensi serta format warnanya.

In [ ]:
unggahan = files.upload()
nama_berkas = list(unggahan.keys())[0]
if len(unggahan) > 1:
    print(f'Anda memilih {len(unggahan)} berkas. Yang dipakai hanya yang pertama: {nama_berkas}')

gambar = Image.open(io.BytesIO(unggahan[nama_berkas]))
print(f'Berkas   : {nama_berkas}')
print(f'Ukuran   : {gambar.size[0]} x {gambar.size[1]} pixel')
print(f'Mode     : {gambar.mode}')

## Sel 5. Klasifikasi dan Visualisasi Hasil

* Menjalankan citra lewat model, memilih kelas dengan skor tertinggi, dan mengevaluasi status pengenalan:
   * Jika skor tertinggi $\ge 0.5$, citra DIKENALI.
   * Jika skor tertinggi $< 0.5$, citra TIDAK DIKENALI.
* Menampilkan 3 panel visualisasi: foto asli, bentuk *grayscale* $32 \times 32$ yang diproses jaringan, serta grafik batang skor *output* ($Y_1, Y_2, Y_3$) terhadap garis ambang batas.

In [ ]:
x = praproses(gambar)
y = umpan_maju(x)

indeks = int(np.argmax(y))
nilai_maks = float(y[indeks])
dikenali = nilai_maks >= AMBANG_DIKENALI

print('=' * 52)
print('HASIL PENGENALAN')
print('=' * 52)
print(f'{"Node output":<14}{"Kelas":<10}{"Nilai":>8}')
for k in range(m):
    tanda = '  <-- terbesar' if k == indeks else ''
    print(f'Y{k + 1:<13}{NAMA_KELAS[k]:<10}{y[k]:>8.3f}{tanda}')
print('-' * 52)

if dikenali:
    print(f'DIKENALI sebagai {NAMA_KELAS[indeks]} (nilai {nilai_maks:.3f}, ambang {AMBANG_DIKENALI})')
else:
    print(f'TIDAK DIKENALI (nilai terbesar {nilai_maks:.3f}, di bawah ambang {AMBANG_DIKENALI})')

if KELAS_SEBENARNYA is not None:
    cocok = (indeks == KELAS_SEBENARNYA) and dikenali
    print(f'Kelas sebenarnya: {NAMA_KELAS[KELAS_SEBENARNYA]} -> {"BENAR" if cocok else "SALAH"}')

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3.2))
a1.imshow(gambar); a1.set_title('Gambar asli'); a1.axis('off')
a2.imshow(x.reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1); a2.set_title('Masuk ke jaringan (grayscale)'); a2.axis('off')
warna = ['tab:green' if k == indeks else 'tab:gray' for k in range(m)]
a3.bar([f'Y{k + 1}\n{NAMA_KELAS[k]}' for k in range(m)], y, color=warna)
a3.axhline(AMBANG_DIKENALI, color='tab:red', linestyle='--', linewidth=1)
a3.set_ylim(0, 1); a3.set_title('Nilai output')
plt.tight_layout()
plt.show()

## Sel 6. Menyimpan Hasil dan Log ke Drive

* Menyimpan hasil pengujian gambar tadi ke folder `HASIL/PENGUJIAN` di Google Drive agar dapat dipakai untuk laporan: log teks, gambar panel hasil, dan salinan gambar yang diuji.
* Menambahkan satu baris ke berkas `RIWAYAT_PENGUJIAN.csv` tiap kali sel ini dijalankan, sehingga semua hasil pengujian terkumpul dalam satu tabel. Nama berkas log dan gambar diberi stempel waktu (UTC) agar tidak saling menimpa.

In [ ]:
import csv
from datetime import datetime, timezone

FOLDER_HASIL = os.path.dirname(os.path.dirname(BERKAS_BOBOT)) + '/HASIL/PENGUJIAN'
os.makedirs(FOLDER_HASIL, exist_ok=True)

waktu = datetime.now(timezone.utc)
stempel = waktu.strftime('%Y%m%d_%H%M%S')
nama_dasar, ekstensi = os.path.splitext(nama_berkas)
awalan = f'{FOLDER_HASIL}/PENGUJIAN_{stempel}_{nama_dasar}'
status = 'DIKENALI' if dikenali else 'TIDAK DIKENALI'

# 1. Salinan gambar yang diuji dan gambar panel hasil
berkas_gambar = awalan + '_GAMBAR' + ekstensi.lower()
with open(berkas_gambar, 'wb') as fh:
    fh.write(unggahan[nama_berkas])
berkas_panel = awalan + '_HASIL.png'
fig.savefig(berkas_panel, dpi=150, bbox_inches='tight')

# 2. Log teks
baris = []
baris.append('LOG PENGUJIAN')
baris.append(f'Waktu (UTC)      : {waktu.strftime("%Y-%m-%d %H:%M:%S")}')
baris.append(f'Berkas bobot     : {BERKAS_BOBOT} ({sumber})')
baris.append(f'Ambang dikenali  : {AMBANG_DIKENALI}')
baris.append('')
baris.append(f'Berkas gambar    : {nama_berkas}')
baris.append(f'Ukuran           : {gambar.size[0]} x {gambar.size[1]} pixel')
baris.append(f'Mode warna       : {gambar.mode}')
baris.append('')
baris.append(f'{"Node output":<14}{"Kelas":<10}{"Nilai":>8}')
for k in range(m):
    tanda = '  <-- terbesar' if k == indeks else ''
    baris.append(f'Y{k + 1:<13}{NAMA_KELAS[k]:<10}{y[k]:>8.3f}{tanda}')
baris.append('')
if dikenali:
    baris.append(f'HASIL: DIKENALI sebagai {NAMA_KELAS[indeks]} (nilai {nilai_maks:.3f}, ambang {AMBANG_DIKENALI})')
else:
    baris.append(f'HASIL: TIDAK DIKENALI (nilai terbesar {nilai_maks:.3f}, di bawah ambang {AMBANG_DIKENALI})')
hasil_cek = ''
if KELAS_SEBENARNYA is not None:
    hasil_cek = 'BENAR' if (indeks == KELAS_SEBENARNYA and dikenali) else 'SALAH'
    baris.append(f'Kelas sebenarnya: {NAMA_KELAS[KELAS_SEBENARNYA]} -> {hasil_cek}')
berkas_log = awalan + '_LOG.txt'
with open(berkas_log, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(baris) + '\n')

# 3. Tambahkan satu baris ke tabel riwayat semua pengujian (CSV)
berkas_riwayat = FOLDER_HASIL + '/RIWAYAT_PENGUJIAN.csv'
baru = not os.path.exists(berkas_riwayat)
with open(berkas_riwayat, 'a', newline='') as fh:
    tulis = csv.writer(fh)
    if baru:
        tulis.writerow(['waktu_utc', 'berkas', 'lebar', 'tinggi', 'mode', 'Y1', 'Y2', 'Y3',
                        'kelas_tertinggi', 'nilai_tertinggi', 'ambang', 'status',
                        'kelas_sebenarnya', 'hasil'])
    tulis.writerow([waktu.strftime('%Y-%m-%d %H:%M:%S'), nama_berkas, gambar.size[0], gambar.size[1],
                    gambar.mode, *[f'{a:.4f}' for a in y], NAMA_KELAS[indeks], f'{nilai_maks:.4f}',
                    AMBANG_DIKENALI, status,
                    NAMA_KELAS[KELAS_SEBENARNYA] if KELAS_SEBENARNYA is not None else '', hasil_cek])

print('=' * 52)
print('HASIL DAN LOG TERSIMPAN')
print('=' * 52)
print(f'Folder: {FOLDER_HASIL}\n')
for b in [berkas_log, berkas_panel, berkas_gambar, berkas_riwayat]:
    print(f'  {os.path.basename(b):<56}{os.path.getsize(b) / 1024:>7.1f} KB')